# Ablation Run

This notebook runs the ablation CV using a **robust baseline pipeline** (imputation + OneHot for categoricals + RandomForest).
It exports `ablation_results.csv`, which is the input for `ablation_inference.ipynb`.


In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor


import sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.ablation_study import run_ablation_cv


In [4]:
# -----------------------
# Configuration
# -----------------------
TRAIN_CSV = os.path.join(PROJECT_ROOT, "Data", "train.csv")  
TARGET_COL = "price"
ID_COL = "carID"

CV_FOLDS = 5
SEED = 42

OUT_RESULTS = os.path.join(PROJECT_ROOT, "ablation_results.csv")


In [5]:
# -----------------------
# Load train
# -----------------------
train_df = pd.read_csv(TRAIN_CSV)

if TARGET_COL not in train_df.columns:
    raise ValueError(f"Expected target column '{TARGET_COL}' in {TRAIN_CSV}")

X_full = train_df.drop(columns=[TARGET_COL, ID_COL], errors="ignore")
y = train_df[TARGET_COL]

print("X_full shape:", X_full.shape)
print("y shape:", y.shape)
print("Columns:", list(X_full.columns))


X_full shape: (75973, 12)
y shape: (75973,)
Columns: ['Brand', 'model', 'year', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']


In [6]:
# -----------------------
# Baseline pipeline builder (RandomForest + OneHot)
# -----------------------

def build_pipeline(use_cols):
    # Infer categorical vs numerical columns from the raw training dataframe
    df = train_df[use_cols]
    cat_cols = [c for c in use_cols if df[c].dtype == "object"]
    num_cols = [c for c in use_cols if c not in cat_cols]

    numeric = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])

    categorical = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric, num_cols),
            ("cat", categorical, cat_cols),
        ],
        remainder="drop",
    )

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    return Pipeline(steps=[("preprocess", pre), ("model", model)])


In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

def run_ablation_cv(X, y, builder, cv_folds=5, random_state=42, baseline_name="Baseline"):
    
    results = []
    
    # Baseline with all features
    pipeline = builder(X.columns.tolist())
    cv = KFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    baseline_scores = cross_val_score(
        pipeline, X, y, 
        cv=cv, 
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    baseline_rmse = np.sqrt(-baseline_scores)
    
    results.append({
        'model': baseline_name,
        'features_removed': 'none',
        'num_features': X.shape[1],
        'mean_rmse': baseline_rmse.mean(),
        'std_rmse': baseline_rmse.std(),
        'fold_scores': baseline_rmse.tolist()
    })
    
    # Ablation: Remove each feature one at a time
    for feature in X.columns:
        X_ablated = X.drop(columns=[feature])
        pipeline = builder(X_ablated.columns.tolist())
        
        scores = cross_val_score(
            pipeline, X_ablated, y,
            cv=cv,
            scoring='neg_mean_squared_error',
            n_jobs=-1
        )
        rmse_scores = np.sqrt(-scores)
        
        results.append({
            'model': f'w/o {feature}',
            'features_removed': feature,
            'num_features': X_ablated.shape[1],
            'mean_rmse': rmse_scores.mean(),
            'std_rmse': rmse_scores.std(),
            'fold_scores': rmse_scores.tolist()
        })
    
    return pd.DataFrame(results)

def build_pipeline(use_cols):
    """
    Build pipeline with RandomForestRegressor + preprocessing
    """
    
    df = train_df[use_cols]
    cat_cols = [c for c in use_cols if df[c].dtype == "object"]
    num_cols = [c for c in use_cols if c not in cat_cols]

    numeric = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])

    categorical = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric, num_cols),
            ("cat", categorical, cat_cols),
        ],
        remainder="drop",
    )

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    return Pipeline(steps=[("preprocess", pre), ("model", model)])

CV_FOLDS = 5
SEED = 42


In [15]:
# -----------------------
# Run ablation CV
# -----------------------
ablation_df = run_ablation_cv(
    X_full, y,
    builder=build_pipeline,
    cv_folds=CV_FOLDS,
    random_state=SEED,
    baseline_name="Baseline"
)

print("Ablation results shape:", ablation_df.shape)
ablation_df.head()


Ablation results shape: (13, 6)


,model,features_removed,num_features,mean_rmse,std_rmse,fold_scores
0,Baseline,none,12,2804.238196,252.064280,"[2569.5535665259463, 2795.9951519468664, 3228...."
1,w/o Brand,Brand,11,2808.611430,249.964259,"[2569.5417160749894, 2802.8354434212306, 3257...."
2,w/o model,model,11,3200.308903,225.392033,"[2991.2510006696766, 3198.862521423136, 3597.3..."
3,w/o year,year,11,3073.533433,217.610384,"[2878.8600955075626, 3072.141488522735, 3425.3..."
4,w/o transmission,transmission,11,2699.353419,205.318529,"[2533.9335120598585, 2666.539829871236, 3053.8..."


In [ ]:

# Save results 

ablation_df.to_csv(OUT_RESULTS, index=False)
print("Saved:", OUT_RESULTS)

# Sanity check: required columns
required_cols = {"model_variant", "features_removed", "fold", "r2", "adjusted_r2", "rmse", "mae", "mse", "mape"}
missing = required_cols - set(ablation_df.columns)
print("Missing columns:", missing)


Saved: d:\Mestrado\MLearn2_2part\ML_group_45-main\ablation_results.csv
Missing columns: {'adjusted_r2', 'rmse', 'mae', 'mse', 'mape', 'fold', 'r2', 'model_variant'}


## Next
Run `ablation_inference.ipynb` with:

- `INPUT_CSV = "ablation_results.csv"`

It should produce:
- `ablation_summary_stats.csv`
- `ablation_significance_tests.csv`
- `ablation_feature_importance.csv`
- `ablation_deploy_check.csv`
